In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-5 ; m=1; R=2; k=10; g=9.8 ;
tf=15; N=int(tf/dt);

t=np.linspace(0,tf,N)


theta_0= 0 ; z_0= 1             #condiciones iniciales (posición)
w_0= 2 ; zdot_0= 0                

## LAGRANGIANO, MOMENTOS Y HAMILTONIANO (H = Σ p·q̇ − L)

In [ ]:
th_s, z_s, thdot_s, zdot_s = sp.symbols('theta z thetadot zdot', real=True)
pth_s, pz_s = sp.symbols('p_theta p_z', real=True)
m_, R_, k_, g_ = sp.symbols('m R k g', positive=True)

# Ligadura x² + y² = R²:  x = R cosθ,  y = R sinθ   (grados de libertad: θ, z)
x_s, y_s = R_*sp.cos(th_s), R_*sp.sin(th_s)
xd_s, yd_s = sp.diff(x_s, th_s)*thdot_s, sp.diff(y_s, th_s)*thdot_s

T_sym = sp.simplify(sp.Rational(1,2)*m_*(xd_s**2 + yd_s**2 + zdot_s**2))
# F = -k r  ->  U = ½ k (x²+y²+z²) ;  gravedad (-z):  m g z
U_sym = sp.simplify(sp.Rational(1,2)*k_*(x_s**2 + y_s**2 + z_s**2) + m_*g_*z_s)

L_sym = T_sym - U_sym
E_sym = T_sym + U_sym

qs, qds, ps = [th_s, z_s], [thdot_s, zdot_s], [pth_s, pz_s]

p_expr = [sp.diff(L_sym, qd) for qd in qds]                              # p_i = ∂L/∂q̇_i   (derivadas parciales)
H_vel  = sp.simplify(sum(p*qd for p, qd in zip(p_expr, qds)) - L_sym)    # H = Σ p_i q̇_i − L   (en función de q, q̇)

print('T =', T_sym)
print('U =', U_sym)
print('p_θ =', p_expr[0], ' ,  p_z =', p_expr[1])
print('H(q, q̇) = Σ p q̇ − L =', H_vel)
print('H − E =', sp.simplify(H_vel - E_sym), '   ->  H = E' if sp.simplify(H_vel - E_sym) == 0 else '   ->  H ≠ E')

# Hamiltoniano en variables canónicas: se invierte p = p(q, q̇)  ->  q̇ = q̇(q, p)
sol   = sp.solve([sp.Eq(P, pe) for P, pe in zip(ps, p_expr)], qds, dict=True)[0]
H_can = sp.simplify(H_vel.subs(sol))
print('H(q, p) =', H_can)

# Ecuaciones de Hamilton:  q̇ = ∂H/∂p ,  ṗ = −∂H/∂q
qdot_eq = [sp.diff(H_can, P) for P in ps]
pdot_eq = [-sp.diff(H_can, q) for q in qs]
print('θ̇ =', qdot_eq[0], ' ,  ż =', qdot_eq[1])
print('ṗ_θ =', pdot_eq[0], ' ,  ṗ_z =', pdot_eq[1])

par_s = (m_, R_, k_, g_)
f_num      = sp.lambdify((th_s, z_s, pth_s, pz_s, *par_s), qdot_eq + pdot_eq, 'numpy')   # (θ̇, ż, ṗ_θ, ṗ_z)
qd_num     = sp.lambdify((th_s, z_s, pth_s, pz_s, *par_s), [sol[thdot_s], sol[zdot_s]], 'numpy')
H_can_num  = sp.lambdify((th_s, z_s, pth_s, pz_s, *par_s), H_can, 'numpy')
H_vel_num  = sp.lambdify((th_s, z_s, thdot_s, zdot_s, *par_s), H_vel, 'numpy')
E_num      = sp.lambdify((th_s, z_s, thdot_s, zdot_s, *par_s), E_sym, 'numpy')
L_num      = sp.lambdify((th_s, z_s, thdot_s, zdot_s, *par_s), L_sym, 'numpy')

## FUNCION PARA EL MOVIMIENTO (ECUACIONES DE HAMILTON)

In [ ]:
def Motion(y0, pars, t):
    """Integra las ecuaciones de Hamilton para (θ, z, p_θ, p_z): Euler predictor + Heun corrector."""
    N = len(t)
    th, z, pth, pz = (np.zeros(N) for _ in range(4))
    th[0], z[0], pth[0], pz[0] = y0

    for i in range(N - 1):
        dt = t[i+1] - t[i]

        # --- PREDICTOR (Euler) ---
        thd_i, zd_i, pthd_i, pzd_i = f_num(th[i], z[i], pth[i], pz[i], *pars)
        th_p  = th[i]  + thd_i*dt
        z_p   = z[i]   + zd_i*dt
        pth_p = pth[i] + pthd_i*dt
        pz_p  = pz[i]  + pzd_i*dt

        # --- Derivadas en el punto predicho ---
        thd_n, zd_n, pthd_n, pzd_n = f_num(th_p, z_p, pth_p, pz_p, *pars)

        # --- CORRECTOR (Heun) ---
        th[i+1]  = th[i]  + 0.5*(thd_i  + thd_n)*dt
        z[i+1]   = z[i]   + 0.5*(zd_i   + zd_n)*dt
        pth[i+1] = pth[i] + 0.5*(pthd_i + pthd_n)*dt
        pz[i+1]  = pz[i]  + 0.5*(pzd_i  + pzd_n)*dt

    return th, z, pth, pz


#Simulación
pars = (m, R, k, g)
pth_0, pz_0 = m*R**2*w_0, m*zdot_0                       # momentos iniciales  p = ∂L/∂q̇
theta, z, pth, pz = Motion((theta_0, z_0, pth_0, pz_0), pars, t)

## COMPROBACIÓN H = E Y GRÁFICAS

In [ ]:
#Comprobación numérica:  H = E  (velocidades por derivación numérica, independientes de H)
thd_g, zd_g = np.gradient(theta, t), np.gradient(z, t)

E  = E_num(theta, z, thd_g, zd_g, *pars)                # E = T + U
Hv = H_vel_num(theta, z, thd_g, zd_g, *pars)            # H = Σ p q̇ − L
Hc = H_can_num(theta, z, pth, pz, *pars)                # H(q, p) integrado con Hamilton

print('max |Σpq̇ − L − E| / |E0| =', np.abs(Hv - E)[1:-1].max()/abs(E[0]))
print('max |H(q,p)  − E| / |E0| =', np.abs(Hc - E)[1:-1].max()/abs(E[0]))

#Gráficas: Lagrangiano, Energía y Hamiltoniano
thd_p, zd_p = qd_num(theta, z, pth, pz, *pars)          # q̇ a partir de los momentos
L   = L_num(theta, z, thd_p, zd_p, *pars)
E_p = E_num(theta, z, thd_p, zd_p, *pars)               # E con esas velocidades (curva suave para dibujar)
s = max(1, N // 20000)                                  # submuestreo

fig, ax = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

ax[0].plot(t[::s], L[::s], color='tab:purple')
ax[0].set(ylabel='L (J)', title='Lagrangiano')

ax[1].plot(t[::s], E_p[::s], color='tab:blue', lw=3, label='E = T + U')
ax[1].plot(t[::s], Hc[::s], 'r--', lw=1.5, label='H(q, p) = Σ p q̇ − L')
ax[1].ticklabel_format(useOffset=False)
ax[1].set_ylim(Hc[0] - 0.5*abs(Hc[0]), Hc[0] + 0.5*abs(Hc[0]))
ax[1].set(ylabel='J', title='H = E y se conservan'); ax[1].legend(fontsize=8)

eps = 1e-18                                             # evita log(0)
ax[2].semilogy(t[::s], np.abs(Hc[::s] - Hc[0])/abs(Hc[0]) + eps, color='tab:red',   lw=1, label='|H − H₀| / |H₀|  (integrador)')
ax[2].semilogy(t[::s], np.abs(Hc[::s] - E_p[::s])/abs(E_p[0]) + eps, color='tab:green', lw=1, label='|H − E| / |E₀|')
ax[2].set(xlabel='t (s)', ylabel='error relativo', title='Error numérico'); ax[2].legend(fontsize=8, loc='lower right')

for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## SOLUCIÓN ANALÍTICA Y COMPARACIÓN

In [ ]:
#Solución analítica
#   θ cíclica  ->  p_θ = m R² θ̇ = cte           ->  θ(t) = θ0 + θ̇0 t
#   m z̈ = -k z - m g  (oscilador armónico)       ->  z(t) = z_eq + (z0 - z_eq) cos(ω0 t) + (ż0/ω0) sin(ω0 t)
om0  = np.sqrt(k/m)                                     # frecuencia del oscilador en z
z_eq = -m*g/k                                           # posición de equilibrio en z

th_an = theta_0 + (pth_0/(m*R**2))*t
z_an  = z_eq + (z_0 - z_eq)*np.cos(om0*t) + (pz_0/(m*om0))*np.sin(om0*t)
E0_an = pth_0**2/(2*m*R**2) + pz_0**2/(2*m) + 0.5*k*(R**2 + z_0**2) + m*g*z_0

print(f'ω0 = {om0:.4f} rad/s,  T_z = {2*np.pi/om0:.4f} s,  z_eq = {z_eq:.4f} m,  θ̇ = {pth_0/(m*R**2):.4f} rad/s')
print(f'max |θ_num − θ_an|     = {np.abs(theta - th_an).max():.2e} rad')
print(f'max |z_num − z_an|     = {np.abs(z - z_an).max():.2e} m')
print(f'max |p_θ − p_θ0|       = {np.abs(pth - pth_0).max():.2e}   (p_θ se conserva)')
print(f'|E_num − E_an| / |E_an| = {np.abs(E[1:-1] - E0_an).max()/abs(E0_an):.2e}')

fig, ax = plt.subplots(2, 2, figsize=(11, 6), sharex=True)

ax[0,0].plot(t[::s], z[::s], lw=2, label='numérica')
ax[0,0].plot(t[::s], z_an[::s], 'r--', lw=1.2, label='analítica')
ax[0,0].axhline(z_eq, color='k', ls=':', lw=1, label=f'z_eq = {z_eq:.3f}')
ax[0,0].set(ylabel='z (m)', title='Coordenada z'); ax[0,0].legend(fontsize=8)

ax[0,1].plot(t[::s], theta[::s], lw=2, label='numérica')
ax[0,1].plot(t[::s], th_an[::s], 'r--', lw=1.2, label='analítica')
ax[0,1].set(ylabel='θ (rad)', title='Ángulo azimutal'); ax[0,1].legend(fontsize=8)

ax[1,0].plot(t[::s], (z - z_an)[::s], color='tab:green')
ax[1,0].set(xlabel='t (s)', ylabel='z_num − z_an')

ax[1,1].plot(t[::s], (theta - th_an)[::s], color='tab:green')
ax[1,1].set(xlabel='t (s)', ylabel='θ_num − θ_an')

for a in ax.ravel(): a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ANIMACIÓN (3D + dos planos 2D)

In [ ]:
#Animación: vista 3D + dos planos 2D (lateral x-z y cenital x-y)
from IPython.display import display
plt.rcParams['animation.embed_limit'] = 100        # MB; el límite por defecto (20) se queda corto

interactivo = True     # True : figura interactiva, se gira la vista 3D con el ratón mientras se anima (requiere ipympl)
                       # False: animación HTML (el ratón no gira la vista; usa 'rotar' para que la cámara gire sola)
elev, azim  = 20, -60  # ángulos iniciales de la vista 3D (grados)
rotar       = 0        # giro automático de la cámara (grados por segundo; 0 = fija). Funciona en los dos modos

if interactivo:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')      # backend interactivo:  pip install ipympl
    except Exception as e:
        interactivo = False
        print('Recordatorio para el modo interactivo (instalar ipympl).', e)

T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 15       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 2.0      # duración de la estela de la partícula (s); 0 = sin estela

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

ths, zs, ts = theta[idx], z[idx], t[idx]
xs, ys = R*np.cos(ths), R*np.sin(ths)              # posición sobre el cilindro

# Límites
Rl = 1.25*R
h  = zs.max() - zs.min() + 1e-9
z0, z1 = min(zs.min(), 0) - 0.15*h, max(zs.max(), 0) + 0.15*h
phi = np.linspace(0, 2*np.pi, 100)

# Muelle de la fuerza F = -k r  (desde el origen O hasta la partícula)
n_coils = 12                                       # vueltas del muelle
amp     = 0.06*R                                   # radio del muelle

def spring_3d(x1, y1, z1_, n=n_coils, amp=amp, npts=300):
    """Hélice 3D que une O con (x1, y1, z1_)."""
    P = np.array([x1, y1, z1_]); d = np.linalg.norm(P); u = P/d       # eje del muelle
    a = np.array([0.0, 0.0, 1.0]) if abs(u[2]) < 0.9 else np.array([1.0, 0.0, 0.0])
    e1 = np.cross(u, a); e1 /= np.linalg.norm(e1); e2 = np.cross(u, e1)   # base perpendicular al eje
    s = np.linspace(0, 1, npts)
    env = np.clip(np.minimum(s, 1 - s)/0.05, 0, 1)                    # extremos rectos
    ang = 2*np.pi*n*s
    pts = np.outer(s*d, u) + amp*env[:, None]*(np.outer(np.cos(ang), e1) + np.outer(np.sin(ang), e2))
    return pts[:, 0], pts[:, 1], pts[:, 2]

def spring_2d(x1, y1, n=n_coils, amp=amp):
    """Zigzag 2D que une O con (x1, y1)."""
    d = max(np.hypot(x1, y1), 1e-9)
    u = np.array([x1, y1])/d                       # vector unitario a lo largo del muelle
    nrm = np.array([-u[1], u[0]])                  # normal
    sgrid = np.linspace(0, 1, 2*n + 1)
    off = amp*np.where(np.arange(2*n + 1) % 2 == 1, 1.0, -1.0)
    off[0] = off[-1] = 0                           # extremos rectos
    pts = np.outer(sgrid*d, u) + np.outer(off, nrm)
    return pts[:, 0], pts[:, 1]

fig = plt.figure(figsize=(14, 4.8))
ax3 = fig.add_subplot(1, 3, 1, projection='3d')
axs = fig.add_subplot(1, 3, 2)                     # plano lateral x-z
axt = fig.add_subplot(1, 3, 3)                     # plano cenital x-y
fig.subplots_adjust(left=0.02, right=0.98, wspace=0.35)

# --- Vista 3D: cilindro ---
ax3.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), zlim=(z0, z1), xlabel='x', ylabel='y', zlabel='z', title='Vista 3D')
ax3.set_box_aspect((2*Rl, 2*Rl, z1 - z0))
ax3.grid(False)                                    # sin rejilla
ax3.view_init(elev=elev, azim=azim)

Zc, Pc = np.meshgrid(np.linspace(z0, z1, 2), np.linspace(0, 2*np.pi, 80))
ax3.plot_surface(R*np.cos(Pc), R*np.sin(Pc), Zc, color='tab:blue', alpha=0.12, linewidth=0, shade=False)   # pared del cilindro
for zc in (z0, z1):
    ax3.plot(R*np.cos(phi), R*np.sin(phi), zc, 'k-', lw=1.2)                                                # bordes (tapas)
for a in np.linspace(0, 2*np.pi, 13)[:-1]:
    ax3.plot([R*np.cos(a)]*2, [R*np.sin(a)]*2, [z0, z1], color='tab:blue', lw=0.6, alpha=0.6)               # generatrices
ax3.plot([0, 0], [0, 0], [z0, z1], 'k:', lw=1)     # eje z
ax3.plot([0], [0], [0], 'ko', ms=4)                # origen O

# --- Plano lateral: paredes del cilindro ---
axs.set(xlim=(-Rl, Rl), ylim=(z0, z1), aspect='equal', xlabel='x (m)', ylabel='z (m)', title='Plano lateral (x-z)')
axs.axvspan(-Rl, -R, color='gray', alpha=0.25); axs.axvspan(R, Rl, color='gray', alpha=0.25)   # exterior del cilindro
axs.axvline(-R, color='k', lw=1.5); axs.axvline(R, color='k', lw=1.5); axs.axhline(0, color='gray', lw=0.5)
axs.plot(0, 0, 'ko', ms=4); axs.grid(alpha=0.3)

# --- Plano cenital: pared del cilindro ---
axt.set(xlim=(-Rl, Rl), ylim=(-Rl, Rl), aspect='equal', xlabel='x (m)', ylabel='y (m)', title='Plano cenital (x-y)')
axt.plot(R*np.cos(phi), R*np.sin(phi), 'k-', lw=1.5)
axt.plot(0, 0, 'ko', ms=4); axt.grid(alpha=0.3)

# Artistas
vec3,   = ax3.plot([], [], [], '-', color='dimgray', lw=1.2)        # muelle O -> partícula
part3,  = ax3.plot([], [], [], 'o', color='tab:red', ms=8)
trail3, = ax3.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.6)

vecS,   = axs.plot([], [], '-', color='dimgray', lw=1.2)
partS,  = axs.plot([], [], 'o', color='tab:red', ms=8)
trailS, = axs.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)

vecT,   = axt.plot([], [], '-', color='dimgray', lw=1.2)
partT,  = axt.plot([], [], 'o', color='tab:red', ms=8)

time_txt = fig.text(0.02, 0.95, '')

def update(k):
    j = max(0, k - trail_n)
    vec3.set_data_3d(*spring_3d(xs[k], ys[k], zs[k]))
    part3.set_data_3d([xs[k]], [ys[k]], [zs[k]])
    trail3.set_data_3d(xs[j:k+1], ys[j:k+1], zs[j:k+1])

    vecS.set_data(*spring_2d(xs[k], zs[k]))
    partS.set_data([xs[k]], [zs[k]])
    trailS.set_data(xs[j:k+1], zs[j:k+1])

    vecT.set_data(*spring_2d(xs[k], ys[k]))
    partT.set_data([xs[k]], [ys[k]])

    if rotar != 0:                                 # solo si se pide giro automático (si no, se respeta la vista del ratón)
        ax3.view_init(elev=elev, azim=azim + rotar*ts[k])

    time_txt.set_text(f't = {ts[k]:.2f} s    θ = {np.degrees(ths[k]) % 360:.1f}°    z = {zs[k]:.2f} m')
    return vec3, part3, trail3, vecS, partS, trailS, vecT, partT, time_txt

anim = animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=False)   # blit=False: necesario con ejes 3D

if interactivo:
    plt.show()                                     # no cerrar la figura: es la que se maneja con el ratón
else:
    plt.close(fig)
    display(HTML(anim.to_jshtml()))